# Parametrsation of gemmae contours


## Documentation and Imports


Created on 27-05-2021

Author: Valentin Laplaud

This code loads gemmae contours from ShapeComputation2.ipynb and express them as a function of a curvilinar abcisse. Adding landmark points (notches and attach points) then allows a deformation of the contour to match the position of landmark points.

In [ ]:
# plotting stuff
import matplotlib as mpl
mpl.use('TkAgg')
%matplotlib inline

COLOR = 'white'
COLOR2 = 'black'

mpl.rcParams['text.color'] = COLOR
mpl.rcParams['axes.labelcolor'] = COLOR
mpl.rcParams['xtick.color'] = COLOR
mpl.rcParams['ytick.color'] = COLOR
mpl.rcParams['axes.edgecolor'] = COLOR

mpl.rcParams["figure.facecolor"] = COLOR2
mpl.rcParams["axes.facecolor"] = COLOR2
mpl.rcParams["savefig.facecolor"] = COLOR2
mpl.rcParams['axes.facecolor'] = COLOR2

from matplotlib import cm
from matplotlib.colors import ListedColormap
import matplotlib.pyplot as plt
import matplotlib.path as mpltPath
from cycler import cycler
#Default colors
colorcycle = [plt.get_cmap('gist_rainbow')(1. * i/30) for i in range(30)]
mpl.rcParams['axes.prop_cycle'] = cycler(color=colorcycle)

# numbers handling
import numpy as np
import pandas as pd
    

# signal processing 
from scipy.signal import savgol_filter, correlate, correlation_lags, find_peaks
from scipy.interpolate import interp1d
from scipy import optimize

# images handling
from skimage import io
from skimage.filters import threshold_otsu, gaussian, laplace, sobel
from skimage.measure import label, regionprops, regionprops_table
from skimage.util import invert
from skimage.morphology import binary_opening, binary_closing, remove_small_holes,binary_erosion
from skimage.color import rgb2gray
from skimage.segmentation import active_contour, morphological_geodesic_active_contour,morphological_chan_vese, checkerboard_level_set, inverse_gaussian_gradient
import cv2 as cv

# to hide known warnings
import warnings
warnings.filterwarnings("ignore")

# General system functions
import os
import shutil
import sys

import time

# my functions
sys.path.append('../')
import VallapFunc as vf



##  Define analysis functions

### Getting landmark points 
User input is used to identify three landmarks points on the first immage (attach point and two notches). Then a curvature computation is used on every subsequent image to get points of high inwards curvature. The position of the three landmarks in time is then propagated from the first image to the nearest point of high curvature.

#### Identifiying contour points from user input.

In [ ]:
def getContourPointsCoordinates(P,s,npts,nimg,X,Y,Title):
            
    RGBimg = io.imread(P + '\\' + s + '.tif', key = nimg) # get the  image from tiff stack

    # ask user to click
    %matplotlib qt
    plt.figure(dpi=250)
    plt.title(Title)
    plt.imshow(RGBimg)
    plt.plot(X,Y,'c',lw = 0.5)
    pts = np.asarray(plt.ginput(npts, timeout=-1))
    plt.close()
    %matplotlib inline

    Points = pts[:]

    for i in range(npts):
        x = pts[i][0]
        y = pts[i][1]

        xc = X[np.argmin(np.sqrt(np.square(X-x)+np.square(Y-y)))]            
        yc = Y[np.argmin(np.sqrt(np.square(X-x)+np.square(Y-y)))]

        Points[i][0] = xc
        Points[i][1] = yc

    return(Points)
 

#### Computing contour curvature unsing circle fitting

In [ ]:
def fitCircle(X,Y):
    
    x_m = np.mean(X)
    y_m = np.mean(Y)    
    
    def calc_R(xc, yc):

        return np.sqrt((X - xc) ** 2 + (Y - yc) ** 2)
    
    def f_2(c):
        
        Ri = calc_R(*c)
        return Ri - Ri.mean()
    
    center_estimate = x_m, y_m
    center_fit, _ = optimize.leastsq(f_2, center_estimate)

    xc_fit, yc_fit = center_fit
    Ri_fit       = calc_R(xc_fit, yc_fit)
    
     #Fitting the radius of the circle
    R_fit        = Ri_fit.mean()
   
    
    if False:
        fig,ax = plt.subplots(dpi=250, facecolor = 'black')
        ax.plot(X,Y,'ro')

        xcircle = xc_fit + R_fit*cos(np.linspace(-pi,pi,100))
        ycircle = yc_fit + R_fit*sin(np.linspace(-pi,pi,100))
        ax.plot(xcircle,ycircle,'k--')

    return([xc_fit, yc_fit,R_fit])
    
def getContourCurvature(X,Y,step):

    newX = np.concatenate((X[-step:],X,X[:step]))    
    newY = np.concatenate((Y[-step:],Y,Y[:step]))
    
    Curv = []
    xc = []
    yc = []
    
    for i in range(len(X)):
        a = fitCircle(newX[i:i+2*step],newY[i:i+2*step])
        xc.append(a[0])
        yc.append(a[1])
        Curv.append(1/a[2])
        
    return(xc,yc,Curv)


#### Smoothing contour and getting landmarks 

In [ ]:
def getLandmarks(CD,GD,StackList,P, **kwargs): 
        
    def round_to_odd(f):
        return np.floor(f) // 2 * 2 + 1    
        
    if not os.path.exists(P + '\\Figures\\'):
        os.mkdir(P + '\\Figures\\') # create folder
        
    if not os.path.exists(P + '\\Figures\\Landmarks\\'):
        os.mkdir(P + '\\Figures\\Landmarks\\') # create folder
            
    DebugPlots = False
    Step = 10
    
    for key, value in kwargs.items(): 
        if key == 'debug':
            DebugPlots = value
        elif key == 'step':
            Step = value
        else:
            print('Unknown key : ' + key + '. Kwarg ignored.')
        
    for s in StackList:

        if not os.path.exists(P + '\\Figures\\Landmarks\\'+ s + '\\'):
            os.mkdir(P + '\\Figures\\Landmarks\\'+ s + '\\') # create folder
        
        for i in range(max(GD.loc[(GD.index == s),'Img'])+1):
            
            Xcenter = GD.loc[(GD.index == s) & (GD['Img'] == i) ,'Xcenter'].values
            Ycenter = GD.loc[(GD.index == s) & (GD['Img'] == i) ,'Ycenter'].values

            Xcont = CD.loc[(CD.index == s) & (CD['Img'] == i) ,'Ximg'].values
            Ycont = CD.loc[(CD.index == s) & (CD['Img'] == i) ,'Yimg'].values

            window = int(round_to_odd(len(Xcont)/50))
            
            Xsmooth = savgol_filter(Xcont, window, 3)
            Ysmooth = savgol_filter(Ycont, window, 3)
            
            CD.loc[(CD.index == s) & (CD['Img'] == i) ,'Xsmooth'] = Xsmooth
            CD.loc[(CD.index == s) & (CD['Img'] == i) ,'Ysmooth'] = Ysmooth
            
            # identifying notches manually on fist image
            if i == 0 :
                
                NotchesRef = getContourPointsCoordinates(P,s,2,i,Xsmooth,Ysmooth,'Click on the two notches (clockwise)')
                AttachRef = getContourPointsCoordinates(P,s,1,i,Xsmooth,Ysmooth,'Click on the attach point')
                
            # Getting data from previous image for every following one
            else:
                
                NotchesRef = [[GD.loc[(GD.index == s) & (GD['Img'] == i-1) ,'Xnotch1'].values,
                               GD.loc[(GD.index == s) & (GD['Img'] == i-1) ,'Ynotch1'].values],
                             [GD.loc[(GD.index == s) & (GD['Img'] == i-1) ,'Xnotch2'].values,
                               GD.loc[(GD.index == s) & (GD['Img'] == i-1) ,'Ynotch2'].values]]
                AttachRef = [[GD.loc[(GD.index == s) & (GD['Img'] == i-1) ,'Xattach'].values, 
                             GD.loc[(GD.index == s) & (GD['Img'] == i-1) ,'Yattach'].values]]
            
            # Automaticaly detect points of high inward curvature            
            xc,yc,Curv = getContourCurvature(Xsmooth,Ysmooth,Step)
            
            # Compute signed curvature using the position of the center of the fitted circle with regards to the contour
            poly = [(x,y) for (x,y) in zip(Xsmooth,Ysmooth)]
            points = [[x,y] for (x,y) in zip(xc,yc)]
            path = mpltPath.Path(poly)            
            isIn = path.contains_points(points)
            
            CurvSign = []
            for Bool in isIn:
                if Bool:
                    CurvSign.append(1)
                else:
                    CurvSign.append(-1)
            
            SignedCurv = np.multiply(Curv,CurvSign)
            
            # Detecting peaks in signed curvature
            loc,prop = find_peaks(-SignedCurv, distance = len(SignedCurv)/15)   
            CurvedPos = Xsmooth[loc],Ysmooth[loc]
            
            # Identifying landmarks    
            xN1 = Xsmooth[loc][np.argmin(np.sqrt(np.square(Xsmooth[loc]-NotchesRef[0][0])+
                                                  np.square(Ysmooth[loc]-NotchesRef[0][1])))]  
            yN1 = Ysmooth[loc][np.argmin(np.sqrt(np.square(Xsmooth[loc]-NotchesRef[0][0])+
                                                  np.square(Ysmooth[loc]-NotchesRef[0][1])))] 
            
            xN2 = Xsmooth[loc][np.argmin(np.sqrt(np.square(Xsmooth[loc]-NotchesRef[1][0])+
                                                  np.square(Ysmooth[loc]-NotchesRef[1][1])))]  
            yN2 = Ysmooth[loc][np.argmin(np.sqrt(np.square(Xsmooth[loc]-NotchesRef[1][0])+
                                                  np.square(Ysmooth[loc]-NotchesRef[1][1])))] 
            
            xA = Xsmooth[loc][np.argmin(np.sqrt(np.square(Xsmooth[loc]-AttachRef[0][0])+
                                                  np.square(Ysmooth[loc]-AttachRef[0][1])))]  
            yA = Ysmooth[loc][np.argmin(np.sqrt(np.square(Xsmooth[loc]-AttachRef[0][0])+
                                                  np.square(Ysmooth[loc]-AttachRef[0][1])))] 
            
            GD.loc[(GD.index == s) & (GD['Img'] == i) ,'Xattach'] = xA
            GD.loc[(GD.index == s) & (GD['Img'] == i) ,'Yattach'] = yA

            GD.loc[(GD.index == s) & (GD['Img'] == i) ,'Xnotch1'] = xN1
            GD.loc[(GD.index == s) & (GD['Img'] == i) ,'Ynotch1'] = yN1

            GD.loc[(GD.index == s) & (GD['Img'] == i),'Xnotch2'] = xN2
            GD.loc[(GD.index == s) & (GD['Img'] == i),'Ynotch2'] = yN2

            # create parametrisation
            npts = len(Xsmooth)
            
            S = np.linspace(0,1,npts)
            
            locn1 = np.argmin(np.abs(np.sqrt(np.square(Xsmooth-xN2)+np.square(Ysmooth-yN2))))
            locn2 = np.argmin(np.abs(np.sqrt(np.square(Xsmooth-xN1)+np.square(Ysmooth-yN1))))
            
            Sn1 = S[locn1]
            Sn2 = S[locn2]
            
            GD.loc[(GD.index == s) & (GD['Img'] == i) ,'LocNotch1'] = locn1
            GD.loc[(GD.index == s) & (GD['Img'] == i) ,'LocNotch2'] = locn2
            GD.loc[(GD.index == s) & (GD['Img'] == i) ,'Snotch1'] = Sn1
            GD.loc[(GD.index == s) & (GD['Img'] == i) ,'Snotch2'] = Sn2

            CD.loc[(CD.index == s) & (CD['Img'] == i) ,'InitialS'] = S
            

            RGBimg = io.imread(P + '\\' + s + '.tif', key = i)

            fig0, [ax0, ax1] = plt.subplots(ncols = 2, dpi = 250,facecolor='black')
            ax0.set_title('Attach point (magenta), notches (red),\n and center (blue) for alignement',fontsize=8)
            ax0.imshow(RGBimg)
            ax0.plot(Xsmooth,Ysmooth,'c',lw = 0.5)
            ax0.plot(xN1,yN1,'.r',ms=2)
            ax0.plot(xN2,yN2,'.r',ms=2)
            ax0.plot(xA,yA,'.m',ms=2)
            ax0.plot(Xcenter[0],Ycenter[0],'*b', ms = 3)
            ax1.set_title(s)
            sc = ax1.scatter(Xsmooth,Ysmooth, c= SignedCurv, cmap = 'BrBG', s = 2)
            ax1.plot(Xsmooth[loc],Ysmooth[loc],'r+',mfc='none', ms = 2, lw= 0.5)
            ax1.set_aspect('equal', adjustable='box')
            #ax1.set_xlim(left=-250+Xcenter[0],right=250+Xcenter[0])
            #ax1.set_ylim(top=250+Ycenter[0],bottom=-250+Ycenter[0])
            fig0.colorbar(sc, ax = ax1, label = 'Curvature',shrink = 0.6)
            fig0.tight_layout()

            fig0.savefig(P + '\\Figures\\Landmarks\\'+ s + '\\' + str(i) +'.png') 
            if DebugPlots & (i == 0):
                plt.show()
            else:
                plt.close()
                
            

    return(CD,GD)

### Reorienting contours and landmarks

Reorientation is done by first centering the contour on the origin, then rotating until alignement of the bissector of the notches-center axes with the horizontal.

In [ ]:
def rotateAndCenterShape(CD,GD,StackList,P,Scale, **kwargs):
    
    if not os.path.exists(P + '\\Figures\\Rotation\\'):
        os.mkdir(P + '\\Figures\\Rotation\\') # create folder
    
    DebugPlots = False
    
    for key, value in kwargs.items(): 
        if key == 'debug':
            DebugPlots = value
        else:
            print('Unknown key : ' + key + '. Kwarg ignored.')
    
    newCD = pd.DataFrame(data=None,columns=['Img','Ximg','Yimg','XAligned','YAligned','Xsmooth','Ysmooth','InitialS','GoodFocus']) 
    
    for s in StackList:
        
        
        if not os.path.exists(P + '\\Figures\\Rotation\\'+ s + '\\'):
            os.mkdir(P + '\\Figures\\Rotation\\'+ s + '\\') # create folder
        
        n = int(1 + np.max(CD.loc[s, 'Img']))
        
        for i in range(n):
            # retrieve important points and contours
            Xc = GD.loc[(GD.index == s) & (GD['Img'] == i), 'Xcenter'].values
            Yc = GD.loc[(GD.index == s) & (GD['Img'] == i), 'Ycenter'].values

            Xcont = CD.loc[(CD.index == s) & (CD['Img'] == i), 'Xsmooth'].values-Xc
            Ycont = CD.loc[(CD.index == s) & (CD['Img'] == i), 'Ysmooth'].values-Yc

            XA = GD.loc[(GD.index == s) & (GD['Img'] == i), 'Xattach'].values-Xc
            YA = GD.loc[(GD.index == s) & (GD['Img'] == i), 'Yattach'].values-Yc

            Xn1 = GD.loc[(GD.index == s) & (GD['Img'] == i), 'Xnotch1'].values-Xc
            Yn1 = GD.loc[(GD.index == s) & (GD['Img'] == i), 'Ynotch1'].values-Yc

            Xn2 = GD.loc[(GD.index == s) & (GD['Img'] == i), 'Xnotch2'].values-Xc
            Yn2 = GD.loc[(GD.index == s) & (GD['Img'] == i), 'Ynotch2'].values-Yc


            A,R = vf.ToCirc([XA,Xn1,Xn2],[YA,Yn1,Yn2],angle='deg')
            Theta,Radius = vf.ToCirc(Xcont,Ycont,angle='deg')
            
            Anotches = np.mod(A[1:3].T-np.array([0,180]),360)
            
            Abis = np.mod(Anotches.sum()/2,360)-180
            
            [XAal,Xn1al,Xn2al],[YAal,Yn1al,Yn2al] = vf.ToCart(A-Abis,R,angle='deg')

            Xcontal,Ycontal = vf.ToCart(Theta-Abis,Radius,angle='deg')

            # Start abcisse from attach points
            locA = np.argmin(np.abs(np.sqrt(np.square(Xcontal-XAal)+np.square(Ycontal-YAal))))
            
            Xcontal = np.roll(Xcontal,-locA)
            Ycontal = np.roll(Ycontal,-locA)

            # Storing contour data
            data = {'Img': i*np.ones(len(Xcontal)), 
                        'Ximg': Xcont+Xc,  
                        'Yimg': Ycont+Yc,
                        'Xsmooth': CD.loc[(CD.index == s) & (CD['Img'] == i), 'Xsmooth'].values,  
                        'Ysmooth': CD.loc[(CD.index == s) & (CD['Img'] == i), 'Ysmooth'].values, 
                        'InitialS': CD.loc[(CD.index == s) & (CD['Img'] == i), 'InitialS'].values,
                        'Xcentered': Xcont,
                        'Ycentered': Ycont,
                        'XAligned': Xcontal*Scale,
                        'YAligned': Ycontal*Scale,
                        'GoodFocus': np.repeat(GD.loc[(GD.index == s) &
                                                             (GD['Img'] == i) ,'GoodFocus'],len(Xcontal))} 
            
            newCD = newCD.append(pd.DataFrame(data=data,index = np.repeat(s,len(Xcont))) )
            
            # create aligned parametrisation
            npts = len(Xcontal)
            
            S = np.linspace(0,1,npts)
            
            locn1al = np.argmin(np.abs(np.sqrt(np.square(Xcontal-Xn2al)+np.square(Ycontal-Yn2al))))
            locn2al = np.argmin(np.abs(np.sqrt(np.square(Xcontal-Xn1al)+np.square(Ycontal-Yn1al))))
            
            Sn1al = S[locn1al]
            Sn2al = S[locn2al]
            
            GD.loc[(GD.index == s) & (GD['Img'] == i) ,'LocNotch1Aligned'] = locn1al
            GD.loc[(GD.index == s) & (GD['Img'] == i) ,'LocNotch2Aligned'] = locn2al
            GD.loc[(GD.index == s) & (GD['Img'] == i) ,'Snotch1Aligned'] = Sn1al
            GD.loc[(GD.index == s) & (GD['Img'] == i) ,'Snotch2Aligned'] = Sn2al
            
            GD.loc[(GD.index == s) & (GD['Img'] == i), 'XattachAligned'] = XAal*Scale
            GD.loc[(GD.index == s) & (GD['Img'] == i), 'YattachAligned'] = YAal*Scale

            GD.loc[(GD.index == s) & (GD['Img'] == i), 'Xnotch1Aligned'] = Xn1al*Scale
            GD.loc[(GD.index == s) & (GD['Img'] == i), 'Ynotch1Aligned'] = Yn1al*Scale

            GD.loc[(GD.index == s) & (GD['Img'] == i), 'Xnotch2Aligned'] = Xn2al*Scale
            GD.loc[(GD.index == s) & (GD['Img'] == i), 'Ynotch2Aligned'] = Yn2al*Scale
        

            fig,[ax0,ax1] = plt.subplots(ncols=2,dpi=200,facecolor='black')
            fig.suptitle(s)
            ax0.plot(Xcont,Ycont,'c')
            ax0.plot(XA,YA,'wo')
            ax0.plot(Xn1,Yn1,'r*',ms=5)
            ax0.plot(Xn2,Yn2,'m*',ms=5)
            ax0.plot(0,0,'b*')
            ax0.plot([0, Xn1],[0, Yn1],'m--')
            ax0.plot([0, Xn2],[0, Yn2],'r--')
            ax0.plot([0, Xn2],[0, Yn2],'r--')
            ax0.set_xlim(left=-250,right=250)
            ax0.set_ylim(top=250,bottom=-250)
            ax0.set_aspect('equal', adjustable='box')

            ax1.plot([-250,250],[0, 0],'w--',lw = 0.5)
            ax1.plot(Xcontal,Ycontal,'c')
            ax1.plot(XAal,YAal,'wo')
            ax1.plot(Xn1al,Yn1al,'r*',ms=5)
            ax1.plot(Xn2al,Yn2al,'m*',ms=5)
            ax1.plot(0,0,'b*')
            ax1.plot([0, Xn1al],[0, Yn1al],'m--')
            ax1.plot([0, Xn2al],[0, Yn2al],'r--')
            ax1.set_xlim(left=-250,right=250)
            ax1.set_ylim(top=250,bottom=-250)
            ax1.set_aspect('equal', adjustable='box')

            fig.tight_layout()
            fig.savefig(P + '\\Figures\\Rotation\\'+ s + '\\' + str(i) +'.png') 
            if DebugPlots & (i == 0):
                plt.show()
            else:
                plt.close()
        
    return(newCD,GD)

### Defining curvilinear-like abscisse with landmarks

In [ ]:
def curvAbsci(CD,GD,StackList,P, **kwargs):

    if not os.path.exists(P + '\\Figures\\Parametrisation\\'):
        os.mkdir(P + '\\Figures\\Parametrisation\\') # create folder
    
    DebugPlots = False
    
    for key, value in kwargs.items(): 
        if key == 'debug':
            DebugPlots = value
        else:
            print('Unknown key : ' + key + '. Kwarg ignored.')
    
    newCD = pd.DataFrame(data=None,columns=['Img','Xparam','Yparam','Sparam','GoodFocus']) 
    
    
    for s in StackList:
        
        if not os.path.exists(P + '\\Figures\\Parametrisation\\'+ s + '\\'):
            os.mkdir(P + '\\Figures\\Parametrisation\\'+ s + '\\') # create folder
            
        n = int(1 + np.max(CD.loc[s, 'Img']))
        
        for i in range(n):
            # retrieve important points and contours
            Xcont = CD.loc[(CD.index == s) & (CD['Img'] == i), 'XAligned'].values
            Ycont = CD.loc[(CD.index == s) & (CD['Img'] == i), 'YAligned'].values
            
            S = CD.loc[(CD.index == s) & (CD['Img'] == i), 'InitialS'].values

            Xat = GD.loc[(GD.index == s) & (GD['Img'] == i), 'XattachAligned'].values
            Yat = GD.loc[(GD.index == s) & (GD['Img'] == i), 'YattachAligned'].values

            Xn1 = GD.loc[(GD.index == s) & (GD['Img'] == i), 'Xnotch1Aligned'].values
            Yn1 = GD.loc[(GD.index == s) & (GD['Img'] == i), 'Ynotch1Aligned'].values

            Xn2 = GD.loc[(GD.index == s) & (GD['Img'] == i), 'Xnotch2Aligned'].values
            Yn2 = GD.loc[(GD.index == s) & (GD['Img'] == i), 'Ynotch2Aligned'].values
            
            locn1 = int(GD.loc[(GD.index == s) & (GD['Img'] == i) ,'LocNotch1Aligned'].values)
            locn2 = int(GD.loc[(GD.index == s) & (GD['Img'] == i) ,'LocNotch2Aligned'].values)
            Sn1 = GD.loc[(GD.index == s) & (GD['Img'] == i) ,'Snotch1Aligned'].values
            Sn2 = GD.loc[(GD.index == s) & (GD['Img'] == i) ,'Snotch2Aligned'].values
            
            # Reparametrisation based on landmarks
            
            VectSa = S[0:locn2+1]
            VectSb = S[locn2+1:locn1+1]
            VectSc = S[locn1+1:]
            
            Sn1_ = 0.75
            Sn2_ = 0.25
            
            VectSa_ = VectSa*Sn2_/Sn2
            VectSb_ = Sn2_ + (VectSb-Sn2)*(Sn1_-Sn2_)/(Sn1-Sn2)
            VectSc_ = Sn1_ + (VectSc-Sn1)*(1-Sn1_)/(1-Sn1)
            
            S_ = np.concatenate((VectSa_,VectSb_,VectSc_))
            
            ##### Plotting
            top = cm.get_cmap('Blues_r', 128)
            bot = cm.get_cmap('Reds', 128)

            newcolors = np.vstack((bot(np.linspace(0, 1, 128)),
                                   top(np.linspace(0, 0.5, 128)),
                                  np.flip(top(np.linspace(0, 0.5, 128)),axis=0),
                                   np.flip(bot(np.linspace(0, 1, 128)),axis=0)
                                  ))
            newcmp = ListedColormap(newcolors, name='RedBlue')

            fig0, [ax,ax2] = plt.subplots(ncols = 2,dpi = 200,facecolor = 'black')
            fig0.suptitle(s)
            ax.set_title('Original')
            sc0 = ax.scatter(Xcont,Ycont,c=S,cmap=newcmp,s=3)
            ax.plot(Xn1,Yn1,'c+',ms=4)
            ax.plot(Xn2,Yn2,'c+',ms=4)
            ax.set_aspect('equal', adjustable='box')
            ax2.set_title('Corrected')
            sc1 = ax2.scatter(Xcont,Ycont,c=S_,cmap=newcmp,s=3)
            ax2.plot(Xn1,Yn1,'c+',ms=4)
            ax2.plot(Xn2,Yn2,'c+',ms=4)
            ax2.set_aspect('equal', adjustable='box')
            fig0.tight_layout()
            cbar0 = fig0.colorbar(sc0, ax=[ax,ax2],ticks=[0, 0.25, 0.5, 0.75, 1],orientation='horizontal')
            cbar0.set_label('Landmark parametrisation')

            fig0.savefig(P + '\\Figures\\Parametrisation\\'+ s + '\\' + str(i) +'.png') 
            if DebugPlots & (i == 0):
                plt.show()
            else:
                plt.close()

            fig,[ax0,ax1] = plt.subplots(nrows = 2,dpi = 200,facecolor = 'black')
            fig.suptitle(s)
            ax0.plot(S,Xcont,'r')
            ax0.plot(S_,Xcont,'b')
            ax0.set_ylabel('X coordinate')
            ax0.legend(['Original parametrisation','Corrected parametrisation'])
            ax1.plot(S,Ycont)
            ax1.plot(S_,Ycont,'b')
            ax1.set_ylabel('Y coordinate')
            ax1.set_xlabel('Landmark parametrisation')

            fig.savefig(P + '\\Figures\\Parametrisation\\'+ s + '\\' + 'XY_' + str(i) +'.png') 
            if DebugPlots & (i == 0):
                plt.show()
            else:
                plt.close()

            
            
            ##########  cut different contour parts in fixed number of points separated by the same distance (per contour parts)
            
            ContourLength = np.sum(np.sqrt(np.square(np.diff(Xcont))+np.square(np.diff(Ycont))))
            ContourCumLength = np.concatenate(([0],np.cumsum(np.sqrt(np.square(np.diff(Xcont))+np.square(np.diff(Ycont))))))

            npts = 250 # number of point per section of contour
            
            # Attach point (0) to first notches clockwise (2)
            SegmentLength_02 = ContourCumLength[locn2]
            deltaS_02 = SegmentLength_02/npts
            SegmentRegCumLength_02 = np.linspace(0,npts,npts+1)*deltaS_02
            keptPtsLoc_02 = [np.argmin(np.abs(ContourCumLength-SegmentRegCumLength_02[i])) for i in range(npts)]

            Xcurv_02 = Xcont[keptPtsLoc_02]
            Ycurv_02 = Ycont[keptPtsLoc_02]
            Scurv_02 = S_[keptPtsLoc_02]
            
            CurvContourLength_02 = np.sum(np.sqrt(np.square(np.diff(Xcurv_02))+np.square(np.diff(Ycurv_02))))
            
            # First notche (2) to second notche (1) clockwise
            SegmentLength_21 = ContourCumLength[locn1]-ContourCumLength[locn2]
            deltaS_21 = SegmentLength_21/(npts*2) # x2 as there is 'half' of the gemma contour between the two notches
            SegmentRegCumLength_21 = np.linspace(0,npts*2,npts*2+1)*deltaS_21+SegmentLength_02
            keptPtsLoc_21 = [np.argmin(np.abs(ContourCumLength-SegmentRegCumLength_21[i])) for i in range(npts*2)]

            Xcurv_21 = Xcont[keptPtsLoc_21]
            Ycurv_21 = Ycont[keptPtsLoc_21]
            Scurv_21 = S_[keptPtsLoc_21]
            
            CurvContourLength_21 = np.sum(np.sqrt(np.square(np.diff(Xcurv_21))+np.square(np.diff(Ycurv_21))))
            
            
            # Second notche (1) to attach pont (0) clockwise
            SegmentLength_10 = ContourCumLength[len(ContourCumLength)-1]-ContourCumLength[locn1]
            deltaS_10 = SegmentLength_10/(npts)
            SegmentRegCumLength_10 = np.linspace(0,npts,npts+1)*deltaS_10+SegmentLength_21+SegmentLength_02
            keptPtsLoc_10 = [np.argmin(np.abs(ContourCumLength-SegmentRegCumLength_10[i])) for i in range(npts)]

            Xcurv_10 = Xcont[keptPtsLoc_10]
            Ycurv_10 = Ycont[keptPtsLoc_10]
            Scurv_10 = S_[keptPtsLoc_10]
            
            CurvContourLength_10 = np.sum(np.sqrt(np.square(np.diff(Xcurv_10))+np.square(np.diff(Ycurv_10))))
            
            print(s + '_' + str(i) + ' contour length variation : ' + 
                  str(round((ContourLength-(CurvContourLength_02+CurvContourLength_21+
                                            CurvContourLength_10+deltaS_02+deltaS_21+deltaS_10))
                            /ContourLength*10000)/100) + '%')
            
            Xcurv = np.concatenate((Xcurv_02, Xcurv_21,Xcurv_10))
            Ycurv = np.concatenate((Ycurv_02, Ycurv_21,Ycurv_10))
            Scurv = np.concatenate((Scurv_02, Scurv_21,Scurv_10))
            
            fig1, ax1 = plt.subplots(dpi = 200,facecolor='black')
            fig1.suptitle(s)
            ax1.set_title("Regular parametrisation by segment (1/10 pts)")
            ax1.plot(Xcont,Ycont,'b-')
            ax1.plot(Xcurv_02[1:len(Xcurv_02)-1:10],Ycurv_02[1:len(Xcurv_02)-1:10],'r*',ms = 1)
            ax1.plot(Xcurv_21[1:len(Xcurv_21)-1:10],Ycurv_21[1:len(Xcurv_21)-1:10],'w*',ms = 1)
            ax1.plot(Xcurv_10[1:len(Xcurv_10)-1:10],Ycurv_10[1:len(Xcurv_10)-1:10],'m*',ms = 1)
            ax1.set_aspect('equal', adjustable='box')
            fig1.savefig(P + '\\Figures\\Parametrisation\\'+ s + '\\' + 'Reg_' + str(i) +'.png') 
            if DebugPlots & (i == 0):
                plt.show()
            else:
                plt.close()
                
            # Contour normalisation
            Thp,Rp = vf.ToCirc(Xcurv,Ycurv)
            XcN,YcN = vf.ToCart(Thp,np.divide(Rp,np.median(Rp)))
            
            # Storing contour data
            data = {'Img': i*np.ones(len(Xcurv)),
                        'Xparam': Xcurv,
                        'Yparam': Ycurv,
                        'XparamNorm': XcN,
                        'YparamNorm': YcN,
                        'Sparam': Scurv,
                        'GoodFocus': np.repeat(GD.loc[(GD.index == s) &
                                                             (GD['Img'] == i) ,'GoodFocus'],len(Xcurv))} 
            
            newCD = newCD.append(pd.DataFrame(data=data,index = np.repeat(s,len(Xcurv))) )            
        
            
    return(newCD)

##  Enter data and run

### Data details

In [ ]:
Scale210318 = 2.27 # Spatial scale (µm/px) 18-03-2021
FPH210318 = 2 # Frames per hour

Scale210319 = 1.72 # Spatial scale (µm/px) 19-03-2021
FPH210319 = 2 # Frames per hour

Scale210528 = 1.94 # Spatial scale (µm/px) 28-05-2021
FPH210528 = 120 # Frames per hour

Scale210618 = 1.72 # Spatial scale (µm/px) 18-06-2021
FPH210618 = 2 # Frames per hour

# Paths to data
P210318 = r'd:\Users\laplaud\Desktop\PostDoc\Data\Microflu\2021.03.18'
P210319 = r'd:\Users\laplaud\Desktop\PostDoc\Data\Microflu\2021.03.19'
P210528 = r'd:\Users\laplaud\Desktop\PostDoc\Data\Microflu\2021.05.28'
P210618 = r'd:\Users\laplaud\Desktop\PostDoc\Data\Microflu\2021.06.18'



# experiments of 18-03-2021 (24hours, image every 30 minutes, brightfield, black background, no Rhizoid)
StackList210318 = ['PPG6','PPG7','PPG9','PPG10','PPG11','PPG15','PPG18','PPG22','PPG23','PPG25','PPG28']  

#StackList210318 = ['PPG18','PPG23']  

# experiments of 19-03-2021 (68h, cropped when air arrives, at least 52 images)
StackList210319 = ['PPG8','PPG10','PPG15','PPG17','PPG18','PPG19','PPG20','PPG21','PPG22']

#StackList210319 = ['PPG17','PPG22']


# experiments of 28-05-2021 (10 minutes, img every 30s. Test whitefield + control no mouvement)
StackList210528 = ['PPG1','PPG2','PPG3','PPG4','PPG5','PPG7','PPG8'
                   ,'PPG9','PPG11','PPG12','PPG13','PPG14']
#StackList210528 = ['PPG13','PPG14']

# experiments of 18-06-2021 (50h, white background test, 1 PPG without flux)

StackList210618 = ['PPG1'] 



print('Data choice made.')


### Loading data 

In [ ]:
## Expe 210528
ContourData210528 = pd.read_csv(P210528 + '\\ContourData210528_Img.csv', index_col = 'Ind')
GlobalData210528 = pd.read_csv(P210528 + '\\GlobalData210528_Img.csv', index_col = 'Ind')    

## Expe 210318
ContourData210318 = pd.read_csv(P210318 + '\\ContourData210318_Img.csv', index_col = 'Ind')
GlobalData210318 = pd.read_csv(P210318 + '\\GlobalData210318_Img.csv', index_col = 'Ind')    

## Expe 210319
ContourData210319 = pd.read_csv(P210319 + '\\ContourData210319_Img.csv', index_col = 'Ind')
GlobalData210319 = pd.read_csv(P210319 + '\\GlobalData210319_Img.csv', index_col = 'Ind')

## Expe 210618
ContourData210618 = pd.read_csv(P210618 + '\\ContourData210618_Img.csv', index_col = 'Ind')
GlobalData210618 = pd.read_csv(P210618 + '\\GlobalData210618_Img.csv', index_col = 'Ind')

print('Loading OK.')

### Shape reparametrisation

#### Landmarks identification

In [ ]:
#ContourData210318_LM, GlobalData210318_LM = getLandmarks(ContourData210318,GlobalData210318,StackList210318,P210318, step = 50, debug = True)
#ContourData210319_LM, GlobalData210319_LM = getLandmarks(ContourData210319,GlobalData210319,StackList210319,P210319, step = 50, debug = True)
ContourData210528_LM,GlobalData210528_LM = getLandmarks(ContourData210528,GlobalData210528,StackList210528,P210528, step = 80, debug = True)
ContourData210618_LM, GlobalData210618_LM = getLandmarks(ContourData210618,GlobalData210618,StackList210618,P210618, step = 50, debug = True)

#### Centering and reorientation

In [ ]:
#ContourData210318_RC,GlobalData210318_RC = rotateAndCenterShape(ContourData210318_LM,GlobalData210318_LM,StackList210318,P210318,Scale210318, debug = True)

#ContourData210319_RC,GlobalData210319_RC = rotateAndCenterShape(ContourData210319_LM,GlobalData210319_LM,StackList210319,P210319,Scale210319, debug = True)

ContourData210528_RC,GlobalData210528_RC = rotateAndCenterShape(ContourData210528_LM,GlobalData210528_LM,StackList210528,P210528,Scale210528, debug = True)

ContourData210618_RC,GlobalData210618_RC = rotateAndCenterShape(ContourData210618_LM,GlobalData210618_LM,StackList210618,P210618,Scale210618, debug = True)

#### Defining curvilinear-like abcisse

In [ ]:
#ContourData210318_Param = curvAbsci(ContourData210318_RC,GlobalData210318_RC,StackList210318,P210318, debug = True)

#ContourData210319_Param = curvAbsci(ContourData210319_RC,GlobalData210319_RC,StackList210319,P210319, debug = True)

ContourData210528_Param = curvAbsci(ContourData210528_RC,GlobalData210528_RC,StackList210528,P210528, debug = True)

ContourData210618_Param = curvAbsci(ContourData210618_RC,GlobalData210618_RC,StackList210618,P210618, debug = True)

### Saving

In [ ]:
#GlobalData210318_RC.to_csv(P210318 + '\\GlobalData210318_Aligned.csv',index_label = 'Ind')
#ContourData210318_RC.to_csv(P210318 + '\\ContourData210318_Aligned.csv',index_label = 'Ind')
#ContourData210318_Param.to_csv(P210318 + '\\ContourData210318_Param.csv',index_label = 'Ind')

#GlobalData210319_RC.to_csv(P210319 + '\\GlobalData210319_Aligned.csv',index_label = 'Ind')
#ContourData210319_RC.to_csv(P210319 + '\\ContourData210319_Aligned.csv',index_label = 'Ind')
#ContourData210319_Param.to_csv(P210319 + '\\ContourData210319_Param.csv',index_label = 'Ind')


GlobalData210528_RC.to_csv(P210528 + '\\GlobalData210528_Aligned.csv',index_label = 'Ind')
ContourData210528_RC.to_csv(P210528 + '\\ContourData210528_Aligned.csv',index_label = 'Ind')
ContourData210528_Param.to_csv(P210528 + '\\ContourData210528_Param.csv',index_label = 'Ind')

GlobalData210618_RC.to_csv(P210618 + '\\GlobalData210618_Aligned.csv',index_label = 'Ind')
ContourData210618_RC.to_csv(P210618 + '\\ContourData210618_Aligned.csv',index_label = 'Ind')
ContourData210618_Param.to_csv(P210618 + '\\ContourData210618_Param.csv',index_label = 'Ind')

## Test Zone